# Suite XCHK — Checking C++ translation units against editions, and as trees

Which editions of C++ have which constructs, as the port records it: `Cpp.Standards` records the editions
`Programs.Cpp` covers as data, C++17, C++20 and C++23, as the user decided ("Maybe C11, C17, C23 Cpp17, cpp20,
cpp23"), each holding the statement, declaration, clause and member kinds it has, as mbse-expressions' C++ dialects
record the terms' (mbse-basis CAP-268); `Cpp.Check.check(store, path, version)` lists the constructs a stored unit uses
that an edition lacks, each located by where it stands in the unit's tree, reading those standards and, for what a
property's value makes, its own `FEATURES`; `Cpp.Check.validate(store, path)` lists a unit's problems as a tree, each
kind's own rules and its terms' as their dialects validate them, what needs an ABI deferred. GNU's extensions are in no
edition, and not checked. Ported from 0.3's C++ cases of CPP (CPP-03 and CPP-04) and PRT.

Each case parses source with `Cpp.Parse.parse` into one store, `Cpp.store()` prepared by `Standards.prepare`, or, where
the parser would not write the tree, writes the unit's `body` by hand in mbse-schemas' builder syntax, as `Parse`
writes it: a list of branches of `Programs.Cpp.Statement`, each tagged by its kind in lower case (`ifstatement`), the
terms nested in it as values of mbse-expressions' unions (`a` below writes the name `a` used as an expression).

In [ ]:
import itertools
from pathlib import Path

from mbse.Expressions.Cpp.Expressions import Expressions as CppExpressions
from mbse.Expressions.Framework import Objects, Standards as ExpressionStandards, Terms
from mbse.Programs import Cpp
from mbse.Programs.Cpp import Check, Parse, Print, Standards
from mbse.Programs.Cpp.Statements import KINDS, LANGUAGE
from mbse.Programs.Framework import Model
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Standards.prepare(Cpp.store())
PATHS = (f"case{n}.cpp" for n in itertools.count())


def parsed(text):
    """The path of a new unit holding `text`, parsed into the store."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return path


def check(text, version):
    return Check.check(STORE, parsed(text), version)


def needs(text, *problems, since="C++20"):
    """`text` is fine in C++23, and has `problems` in the editions before it: from C++17 to the one before `since`, all
    of them; in C++20, those that need C++23."""
    path = parsed(text)
    assert Check.check(STORE, path, "C++23") == [], Check.check(STORE, path, "C++23")
    assert Check.check(STORE, path, "C++17") == list(problems), Check.check(STORE, path, "C++17")
    later = [p for p in problems if p.endswith("C++23")] if since == "C++20" else list(problems)
    assert Check.check(STORE, path, "C++20") == later, Check.check(STORE, path, "C++20")


def written(*body):
    """The path of a new unit whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.Cpp.Statement`, as `Parse.parse` writes one."""
    path = next(PATHS)

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "unit": txn.Programs.Cpp.TranslationUnit})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.unit)}),
                lambda _: _.unit.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return path


def validate(*body):
    return Check.validate(STORE, written(*body))


# Terms and statements the cases share, in the builders' syntax: three names, a name `x` and `int`; `;` and `b;`
NAME = lambda n: n.identifier(lambda i: i.spelling("x"))  # noqa: E731
a = lambda t: t.idexpression(lambda i: i.name(lambda n: n.identifier(lambda x: x.spelling("a"))))  # noqa: E731
b = lambda t: t.idexpression(lambda i: i.name(lambda n: n.identifier(lambda x: x.spelling("b"))))  # noqa: E731
c = lambda t: t.idexpression(lambda i: i.name(lambda n: n.identifier(lambda x: x.spelling("c"))))  # noqa: E731
INT = lambda p: p.primitivetypespecifier(lambda k: k.keyword("int"))  # noqa: E731
EMPTY = lambda u: u.expressionstatement(lambda s: s)  # noqa: E731
SAY_B = lambda u: u.expressionstatement(lambda s: s.expression(b))  # noqa: E731
IS = lambda t: lambda c: c.expression(t)  # noqa: E731  # a condition that declares nothing: a term

## XCHK-01 · The editions as data: `Programs.Cpp` covers C++17, C++20 and C++23, each holding its statement, declaration, clause and member kinds, beside mbse-expressions' dialects' own; run again, nothing changes

In [ ]:
DESIGNATIONS = ["ISO/IEC 14882:2017", "ISO/IEC 14882:2020", "ISO/IEC 14882:2024"]
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    standards = ExpressionStandards.standards(language)
    assert list(standards) == DESIGNATIONS
    assert ExpressionStandards.find(language, "C++23") is standards["ISO/IEC 14882:2024"]
    assert ExpressionStandards.properties(standards["ISO/IEC 14882:2020"]) == {
        "bodies": ("org.iso", "ch.iec"), "number": "14882", "revision": ("year", 2020), "names": ("C++20",)}

    def kinds(designation):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(standards[designation])}

    assert len(KINDS) == 44 and kinds(DESIGNATIONS[2]) == set(KINDS) - {"AsmDeclaration"}  # GNU's `asm`, in none
    assert kinds(DESIGNATIONS[1]) == kinds(DESIGNATIONS[2])  # C++23 adds no kind of its own
    assert kinds(DESIGNATIONS[2]) - kinds(DESIGNATIONS[0]) == set(Standards.SINCE)  # C++20's
    assert "Error" not in kinds(DESIGNATIONS[2])  # invalid syntax, in none
    assert {"IfStatement", "Handler", "AccessSpecifier"} <= kinds(DESIGNATIONS[0])  # C's kinds C++'s, a clause, a member
    for dialect in Standards.DIALECTS[1:]:  # the terms' editions are mbse-expressions' own, by the same designations
        covered = list(ExpressionStandards.standards(STORE.singleton(dialect)))
        assert [d for d in covered if d in standards] == list(standards), dialect  # C Preprocessor's beside C's
    # A directive kind's edition, read from data: C Preprocessor's C++ editions, `#elifdef` C++23's, `__has_include`
    # C++17's
    preprocessor = ExpressionStandards.standards(STORE.singleton("Expressions.C.Preprocessor"))

    def directives(designation):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(preprocessor[designation])}

    assert "ElifdefDirective" in directives(DESIGNATIONS[2]) and "ElifdefDirective" not in directives(DESIGNATIONS[1])
    assert "HasIncludeExpression" in directives(DESIGNATIONS[0])
    expressions = ExpressionStandards.standards(STORE.singleton("Expressions.Cpp.Expressions"))
    tags = {kind.name for kind in ExpressionStandards.kinds(expressions["ISO/IEC 14882:2017"])}
    assert "Expressions.Cpp.Expressions.LambdaExpression" in tags
    assert "Expressions.Cpp.Expressions.RequiresExpression" not in tags  # C++20's
# The language is one of the framework's dialects, which is what may own a standard
dialects = STORE.registered("Expressions.Framework.Dialect")
assert any(b.name == "programs_cpp" and b.schema.name == "Programs.Cpp.Language" for b in dialects.branches)
# Run again, each writes nothing: the same editions, each with the same kinds
Standards.prepare(STORE)
with STORE.transaction():
    again = ExpressionStandards.standards(STORE.singleton(LANGUAGE))
    assert list(again) == list(standards) and len(ExpressionStandards.kinds(again["ISO/IEC 14882:2024"])) == 43
assert Standards.EXTENSIONS == ("AsmDeclaration",) and Standards.EDITIONS["C++20"] == ("ISO/IEC 14882:2020", 2020)
# What `define` writes from `SINCE`, `check` reads: a kind recorded as arriving later is lacking before
Standards.SINCE["StaticAssertDeclaration"] = "C++23"
try:
    later = Standards.prepare(Cpp.store())
finally:
    del Standards.SINCE["StaticAssertDeclaration"]
later.transaction(Parse.parse("static_assert(1, \"m\");\n", "later.cpp"))
assert Check.check(later, "later.cpp", "C++23") == []
assert Check.check(later, "later.cpp", "C++20") == ["body[0]: StaticAssertDeclaration needs C++23"]
print("ok")

## XCHK-02 · Where features exist: C++20's and C++23's, each located by where it stands in the unit's tree, each once; what C++17 has, every edition has

In [ ]:
# The language's data: C++20's kinds
needs("module m;\nexport import n;\nexport int x;\nusing enum E;\nauto t() { co_return; }\n"
      "template <class T> concept C = true;\n",
      "body[0]: ModuleDeclaration needs C++20", "body[1]: ImportDeclaration needs C++20",
      "body[2]: ExportDeclaration needs C++20", "body[3]: UsingEnumDeclaration needs C++20",
      "body[4].body.body[0]: CoReturnStatement needs C++20", "body[5].declaration: ConceptDefinition needs C++20")
# The dialects' data: C++20's terms
needs("auto f() { co_await x; co_yield 1; return requires { x; }; }\nauto p = P{.x = 1};\n",
      "body[0].body.body[0].expression: awaitexpression needs C++20",
      "body[0].body.body[1].expression: yieldexpression needs C++20",
      "body[0].body.body[2].value: requiresexpression needs C++20",
      "body[1].declarators[0].initializer: designatedinitializer needs C++20")
# Statements, declarations and parts, `STATEMENT_FEATURES`
needs("void f() {\n    if consteval {}\n    for (using T = int; ;) {}\n    for (int i = 0; int x : v) {}\nl:\n}\n",
      "body[0].body.body[0]: IfStatement.consteval needs C++23",
      "body[0].body.body[1]: ForStatement with an alias declaration as its init-statement needs C++23",
      "body[0].body.body[2]: RangeForStatement.initializer needs C++20",
      "body[0].body.body[3]: LabeledStatement without a statement needs C++23")
needs("void f() {\n    switch (x) {\n    case 1:\n    }\nm:\n    int y = 0;\n}\nint g() requires true;\n"
      "template <class T> requires true void h();\nstruct S { int x : 4 = 1; void k() requires true; };\n"
      "namespace a::inline b {}\n",
      "body[0].body.body[0].body.body[0]: CaseStatement without a statement needs C++23",
      "body[0].body.body[1]: LabeledStatement before a declaration needs C++23",
      "body[1].declarators[0]: InitDeclarator.requires needs C++20", "body[2]: TemplateDeclaration.requires needs C++20",
      "body[3].specifiers[0].members.members[0].declarators[0]: MemberDeclarator with a bit-field's initializer needs "
      "C++20", "body[3].specifiers[0].members.members[1].declarators[0]: MemberDeclarator.requires needs C++20",
      "body[4]: NamespaceDefinition with an inline nested name needs C++20")
needs("void f() {\n    if (using T = int; true) {}\n    switch (using U = int; 1) {}\n"
      "    for (using V = int; int x : v) {}\n}\n",
      "body[0].body.body[0]: IfStatement with an alias declaration as its init-statement needs C++23",
      "body[0].body.body[1]: SwitchStatement with an alias declaration as its init-statement needs C++23",
      "body[0].body.body[2]: RangeForStatement with an alias declaration as its init-statement needs C++23")
needs("struct S {\n    static int operator()(int x) { return x; }\n    static int operator[](int i);\n};\n"
      "template <class T> void f() requires true {}\n",
      "body[0].specifiers[0].members.members[0]: FunctionDefinition static operator() needs C++23",
      "body[0].specifiers[0].members.members[1]: MemberDeclaration static operator[] needs C++23",
      "body[1].declaration: FunctionDefinition.requires needs C++20")
# Terms and helpers, `FEATURES`
needs("consteval int f() { return 1; }\nconstinit int g = 1;\nchar8_t c = u8'c';\nauto o = 1 <=> 2;\n"
      "struct S { auto operator<=>(const S &) const = default; };\n",
      "body[0].specifiers[0]: declspecifier.keyword 'consteval' needs C++20",
      "body[1].specifiers[0]: declspecifier.keyword 'constinit' needs C++20",
      "body[2].specifiers[0]: primitivetypespecifier.keyword 'char8_t' needs C++20",
      "body[3].declarators[0].initializer: binaryexpression.operator '<=>' needs C++20",
      "body[4].specifiers[0].members.members[0].declarator: operatorname.operator '<=>' needs C++20")
needs("auto l = []<class T>(T x) requires true { return x; };\nauto m = [] [[nodiscard]] () static { return 1; };\n"
      "auto n = [...x = 1] {};\n",
      "body[0].declarators[0].initializer: lambdaexpression.template_parameters needs C++20",
      "body[0].declarators[0].initializer: lambdadeclarator.requires needs C++20",
      "body[1].declarators[0].initializer: lambdaexpression.attributes needs C++23",
      "body[1].declarators[0].initializer: lambdadeclarator static needs C++23",
      "body[2].declarators[0].initializer: initcapture.pack needs C++20")
needs("void f(C auto x);\ntemplate <template <class> class U> requires true struct W;\nvoid g(this auto &self);\n"
      "template <class T> requires true struct V;\n",
      "body[0].declarators[0].declarator: placeholdertypespecifier.constraint needs C++20",
      "body[1]: TemplateDeclaration.requires needs C++20",
      "body[2].declarators[0].declarator: parameterdeclaration.this_keyword needs C++23",
      "body[3]: TemplateDeclaration.requires needs C++20")
needs("int a = m[1, 2] + auto(x) + 1z + 2uz;\n#warning \"w\"\n",
      "body[0].declarators[0].initializer: subscriptexpression of several indices needs C++23",
      "body[0].declarators[0].initializer: functionalcastexpression of auto needs C++23",
      "body[0].declarators[0].initializer: literal.suffix 'z' needs C++23",
      "body[0].declarators[0].initializer: literal.suffix 'uz' needs C++23",
      "body[1].directive: #warning needs C++23")  # a directive's name within `otherdirective`, which no data records
# The dialects' data: C Preprocessor's C++ editions, `#elifdef` and `#elifndef` C++23's, once; `__has_include` C++17's
needs("#ifdef A\n#elifdef B\n#elifndef C\n#endif\n#if __has_include(<x>)\n#endif\n",
      "body[0].directive: elifdefdirective needs C++23")
assert not {"elifdefdirective", "hasincludeexpression"} & set(Check.FEATURES)  # data, not code
needs("struct S { explicit(true) S(int); explicit S(); };\nauto [p, q] = r;\n",
      "body[0].specifiers[0].members.members[0].specifiers[0]: explicitspecifier.condition needs C++20")  # C++17's
# structured bindings in every edition
# A type parameter's constraint and a template template parameter's `requires`, written: `Parse` reads
# `template <C T>` as a constant's parameter (its departure 46), and tree-sitter no template template parameter's
# `requires`
T = lambda i: i.spelling("T")  # noqa: E731
path = written(lambda u: u.templatedeclaration(lambda t: t.parameters([
    lambda p: p.typeparameter(lambda k: k.key("class").constraint(lambda n: n.identifier(lambda i: i.spelling(
        "C"))).name(T)),
    lambda p: p.templatetemplateparameter(lambda k: k.key("class").requires(a).name(T))]).declaration(
        lambda d: d.simpledeclaration(lambda s: s.specifiers([INT]).declarators([lambda d: d.declarator(
            lambda k: k.iddeclarator(lambda i: i.name(NAME)))])))))
assert Check.check(STORE, path, "C++17") == ["body[0].parameters[0]: typeparameter.constraint needs C++20",
                                             "body[0].parameters[1]: templatetemplateparameter.requires needs C++20"]
# What C++17 has, every edition has: its own features, C++11's and C++14's, and GNU's extensions, in none
assert check("auto f() { if constexpr (true) {} if (int x = 1; x) {} switch (int y = 1; y) {} return 0; }\n"
             "template <class... T> auto sum(T... t) { return (t + ...); }\nnamespace a::b {}\n"
             "using std::a, std::b;\nauto [[nodiscard]] g() -> int;\n[[using gnu: hot]] void h();\n"
             "inline int v = 1;\nauto l = [*this, x = 1] { return x; };\nstruct S { int y = 0; };\n"
             "template <class T> struct V; V(int) -> V<int>;\nstatic_assert(true);\nchar u = u8'c';\n"
             "int k = ({ 1; }) + __extension__ 2;\nint x __attribute__((aligned(4)));\n", "C++17") == []
print("ok")

## XCHK-03 · The edition by its number, its common name or its designation; editions the language does not cover, a store not prepared, and a path the workspace does not hold

In [ ]:
path = parsed("template <class T> concept C = true;\n")
problems = ["body[0].declaration: ConceptDefinition needs C++20"]
for version in ((17,), "C++17", "ISO/IEC 14882:2017"):
    assert Check.check(STORE, path, version) == problems, version
for version in ((20,), "C++20", "ISO/IEC 14882:2020", (23,), "C++23", "ISO/IEC 14882:2024"):
    assert Check.check(STORE, path, version) == [], version
for version in ((11,), (2020,), (1, 1), "C++11", "C23", "ISO/IEC 9899:2024", "2020"):
    with raises(LookupError, match="Programs.Cpp covers no edition"):
        Check.check(STORE, path, version)
with raises(LookupError):
    Check.check(STORE, "nowhere.cpp", "C++23")
with raises(LookupError):
    Check.validate(STORE, "nowhere.cpp")
# Within the caller's transaction, a unit's visitor is read; the store and a path open their own
with STORE.transaction():
    unit = STORE.singleton(Model.WORKSPACE).files({"path": path})
    assert Check.check_file(STORE, unit, "C++17") == problems
    assert Check.validate_file(unit) == []
    with raises(ValueError, match="transactions do not nest"):
        Check.check(STORE, path, "C++17")
# A store that holds the installs alone has no editions; one with the language's alone has no terms' editions
bare = Cpp.store()
bare.transaction(Parse.parse("template <class T> concept C = true;\n", "a.cpp"))
with raises(LookupError, match="run Standards.prepare"):
    Check.check(bare, "a.cpp", "C++23")
bare.transaction(Standards.install)
bare.transaction(Standards.define)
with raises(LookupError, match="Expressions.Cpp.Expressions covers no edition"):
    Check.check(bare, "a.cpp", "C++23")
assert Check.check(Standards.prepare(bare), "a.cpp", "C++23") == []
print("ok")

## XCHK-04 · The conformance sources: none of their constructs lacking in C++23; in C++20 and C++17, each of the later editions' where it stands; valid

In [ ]:
path = parsed((SOURCES / "cpp20.cpp").read_text(encoding="utf-8"))
assert Check.check(STORE, path, "C++23") == Check.check(STORE, path, "C++20") == Check.check(STORE, path, (20,)) == []
oldest = Check.check(STORE, path, "C++17")
assert oldest[:3] == ["body[2]: GlobalModuleFragment needs C++20", "body[11]: ModuleDeclaration needs C++20",
                      "body[12]: ImportDeclaration needs C++20"], oldest
assert "body[55].body.body[46]: RangeForStatement.initializer needs C++20" in oldest
assert "body[35].declaration.specifiers[0].members.members[5].specifiers[0]: explicitspecifier.condition needs C++20" in (
    oldest)
assert len(oldest) == 26 and all(problem.endswith("needs C++20") for problem in oldest), oldest
assert Check.validate(STORE, path) == []  # its structured bindings and `explicit(N > 1)` among them
path = parsed((SOURCES / "cpp23.cpp").read_text(encoding="utf-8"))
assert Check.check(STORE, path, "C++23") == []
newest = Check.check(STORE, path, "C++20")
assert len(newest) == 11 and all(problem.endswith("needs C++23") for problem in newest), newest
assert "body[12].body.body[0]: IfStatement.consteval needs C++23" in newest
oldest = Check.check(STORE, path, "C++17")
assert len(oldest) == 19 and set(newest) < set(oldest), oldest
assert Check.validate(STORE, path) == []
print(len(oldest), "constructs of C++23's source lacking in C++17")

## XCHK-05 · Each kind's own rules: what it needs, what its lists must hold, what goes together, the words its properties may be, and the dangling `else`; a declaration needs no type specifier in C++

In [ ]:
DECLARE_X = lambda d: d.declarator(lambda k: k.iddeclarator(lambda i: i.name(NAME)))  # noqa: E731
FUNCTION_X = lambda k: k.functiondeclarator(lambda f: f.declarator(lambda k: k.iddeclarator(  # noqa: E731
    lambda i: i.name(NAME))))
BLOCK = lambda k: k  # noqa: E731  # a compound statement, which a `try` and a handler hold alone
problems = [
    (lambda u: u.comment(lambda s: s), "body[0]: a Comment needs a text"),
    (lambda u: u.ifstatement(lambda s: s.consequence(EMPTY)), "body[0]: an IfStatement needs a condition"),
    (lambda u: u.ifstatement(lambda s: s.condition(IS(a))), "body[0]: an IfStatement needs a consequence"),
    (lambda u: u.ifstatement(lambda s: s.negated(True).condition(IS(a)).consequence(EMPTY)),
     "body[0]: an IfStatement negated needs consteval"),
    (lambda u: u.rangeforstatement(lambda s: s.range(a).body(EMPTY)),
     "body[0]: a RangeForStatement needs a declaration"),
    (lambda u: u.trystatement(lambda s: s.body(BLOCK)), "body[0]: a TryStatement needs a handler"),
    (lambda u: u.trystatement(lambda s: s.body(BLOCK).handlers([lambda h: h.body(BLOCK)])),
     "body[0].handlers[0]: a Handler needs a parameter"),
    (lambda u: u.gotostatement(lambda s: s), "body[0]: a GotoStatement needs a label or a target"),
    (lambda u: u.usingdeclaration(lambda s: s), "body[0]: a UsingDeclaration needs a declarator"),
    (lambda u: u.importdeclaration(lambda s: s.export(True)),
     "body[0]: an ImportDeclaration needs a name, a partition or a header"),
    (lambda u: u.staticassertdeclaration(lambda s: s.keyword("_Static_assert").condition(a)),
     "body[0]: StaticAssertDeclaration.keyword cannot be '_Static_assert'"),  # C's spelling, not C++'s
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.declspecifier(lambda k: k.keyword(
        "explicit"))]).declarators([DECLARE_X])),
     "body[0].specifiers[0]: explicit is an ExplicitSpecifier, not a DeclSpecifier"),  # mbse-expressions' own check
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("class").body(
        True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.accessspecifier(lambda a: a.access(
            "friend"))]))))])),
     "body[0].specifiers[0].members.members[0]: AccessSpecifier.access cannot be 'friend'"),
    (lambda u: u.templatedeclaration(lambda t: t), "body[0]: a TemplateDeclaration needs a declaration"),
    (lambda u: u.functiondefinition(lambda f: f.declarator(FUNCTION_X)), "body[0]: a FunctionDefinition needs a body"),
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("struct").body(
        True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.memberdeclaration(lambda d: d.specifiers(
            [INT]).declarators([lambda d: d]))]))))])),
     "body[0].specifiers[0].members.members[0].declarators[0]: a MemberDeclarator needs a declarator or a bitfield"),
]
for body, problem in problems:
    assert validate(body) == [problem], (problem, validate(body))
assert validate(lambda u: u.ifstatement(lambda s: s.consteval(True).consequence(EMPTY))) == []  # `if consteval`
assert validate(lambda u: u.simpledeclaration(lambda s: s.declarators([DECLARE_X]))) == []  # no type specifier
assert validate(lambda u: u.functiondefinition(lambda f: f.declarator(FUNCTION_X).body(
    lambda k: k.defaultedbody(lambda d: d)))) == []  # `x() = default`, a constructor's
assert validate(lambda u: u.functiondefinition(lambda f: f.specifiers([INT]).declarator(lambda k: k.iddeclarator(
    lambda i: i.name(NAME))).body(lambda k: k.compoundstatement(lambda k: k)))) == [
    "body[0]: a FunctionDefinition's declarator declares no function"]
# `if (a) for (int x : v) if (b) ; else ;` would give the else to the inner `if`
dangling = "body[0]: an if without else ends the consequence, and would take the else"
inner = lambda k: k.rangeforstatement(lambda f: f.declaration(lambda d: d.specifiers([INT]).declarators([  # noqa: E731
    DECLARE_X])).range(c).body(lambda k: k.ifstatement(lambda s: s.condition(IS(b)).consequence(EMPTY))))
assert validate(lambda u: u.ifstatement(lambda s: s.condition(IS(a)).consequence(inner).alternative(EMPTY))) == [dangling]
assert validate(lambda u: u.ifstatement(lambda s: s.condition(IS(a)).consequence(lambda k: k.dostatement(
    lambda d: d.condition(b).body(EMPTY))).alternative(EMPTY))) == []
assert validate(lambda u: u.ifstatement(lambda s: s.condition(IS(a)).consequence(lambda k: k.ifstatement(
    lambda s: s.condition(IS(b)).consequence(EMPTY).alternative(lambda k: k.whilestatement(lambda w: w.condition(
        IS(c)).body(lambda k: k.ifstatement(lambda s: s.condition(IS(c)).consequence(EMPTY))))))).alternative(
    EMPTY))) == [dangling]  # through its alternative
assert validate(lambda u: u.ifstatement(lambda s: s.condition(IS(a)).consequence(lambda k: k.labeledstatement(
    lambda l: l.label(lambda i: i.spelling("l")))).alternative(EMPTY))) == []  # a label without a statement (C++23)
print(len(problems), "problems")

## XCHK-06 · Terms as their dialects validate them, located at the outermost term, helper or part that holds them; a helper a statement holds alone, by its own checks; names taken as bound; what needs an ABI deferred

In [ ]:
BAD = lambda t: t.memberexpression(lambda m: m.object(a).operator("::").member(  # noqa: E731
    lambda n: n.identifier(lambda i: i.spelling("m"))))
assert validate(lambda u: u.expressionstatement(lambda s: s.expression(BAD))) == [
    "body[0].expression: a memberexpression's operator must be '.' or '->', got '::'"]
assert validate(lambda u: u.returnstatement(lambda s: s.value(lambda t: t.idexpression(lambda i: i.name(
    lambda n: n.identifier(lambda x: x.spelling("1x"))))))) == ["body[0].value: '1x' is not an identifier"]
assert validate(lambda u: u.simpledeclaration(lambda s: s.specifiers([INT]).declarators([lambda d: d.declarator(
    lambda k: k.iddeclarator(lambda i: i.name(NAME))).initializer(lambda i: i.equalinitializer(lambda e: e.value(
        BAD)))]))) == [
    "body[0].declarators[0].initializer: a memberexpression's operator must be '.' or '->', got '::'"]  # in a part
assert validate(lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.namedtypespecifier(
    lambda n: n.name(lambda n: n.identifier(lambda i: i.spelling("2t"))))]).declarators([lambda d: d.declarator(
        lambda k: k.arraydeclarator(lambda r: r.declarator(lambda k: k.iddeclarator(lambda i: i.name(NAME))).size(
            BAD)))]))) == [
    "body[0].specifiers[0]: '2t' is not an identifier",
    "body[0].declarators[0].declarator: a memberexpression's operator must be '.' or '->', got '::'"]
assert validate(lambda u: u.directive(lambda s: s.directive(lambda d: d.includedirective(lambda i: i.directive(
    "include"))))) == ["body[0].directive: an IncludeDirective has a path or a macro"]  # C Preprocessor's own
# Parsed units: names are the unit's to bind; a type C++ leaves to the implementation is not checked against its
# range, which only an ABI gives
assert Check.validate(STORE, parsed("int f(int a) { return a + b + g(c); }\nchar c = 300;\nint i = 1u;\n"
                                    "auto s = sizeof(long double) + alignof(wchar_t);\n")) == []
print("ok")

## XCHK-07 · Printing writes what the store holds; checking it against an edition is `check`'s, and validating it `validate`'s (ported from PRT)

In [ ]:
path = parsed("template <class T> concept C = true;\n")
assert Print.print_module(STORE, path) == "template <class T>\nconcept C = true;\n"  # printed, whatever the edition
assert Check.check(STORE, path, "C++17") == ["body[0].declaration: ConceptDefinition needs C++20"]
path = written(lambda u: u.ifstatement(lambda s: s.condition(IS(a)).consequence(lambda k: k.ifstatement(
    lambda s: s.condition(IS(b)).consequence(SAY_B))).alternative(SAY_B)))
assert Print.print_module(STORE, path) == "if (a)\n    if (b)\n        b;\nelse\n    b;\n"  # printed as held
assert Check.validate(STORE, path) == ["body[0]: an if without else ends the consequence, and would take the else"]
print("ok")

## XCHK-08 · Invalid syntax, which a parse keeps: `validate` reports each `Error` and each `error` and `missing` term, located; `check` skips them and checks what was recovered within an `Error`

In [ ]:
path = parsed("int a = 1;\n"
              "void f() {\n"
              "    int x = = 1;\n"
              "    int y = (a + );\n"
              "}\n"
              "struct s { int = ; int ok; };\n"
              "x = 1;\n")
assert Check.validate(STORE, path) == [
    "body[1].body.body[0]: invalid syntax 'int x = = 1;'",
    "body[1].body.body[1].declarators[0].initializer: expression: right: missing identifier",
    "body[2].specifiers[0].members.members[0]: invalid syntax 'int = ;'",  # where a member stands
    "body[3]: invalid syntax 'x = 1;'",  # a statement at namespace scope, which C++ does not have
], Check.validate(STORE, path)
assert Check.check(STORE, path, "C++17") == []  # skipped: no edition lists invalid syntax
recovered = parsed("void f() {\n    if (a {}\n    co_return;\n}\n")
assert Check.validate(STORE, recovered) == ["body[0].body.body[0]: invalid syntax 'if (a {}\\n    co_return;'"]
assert Check.check(STORE, recovered, "C++17") == ["body[0].body.body[0].body[0]: CoReturnStatement needs C++20"]
print("ok")

## XCHK-09 · Statements within terms: a lambda's `block`, a class's `members` and a conditional directive's `group`, checked and validated as statements, located through the term's place, to any depth; mbse-expressions' dialects do not read them

In [ ]:
path = parsed("auto l = [] {\n"
              "    co_return;\n"
              "};\n"
              "struct S {\n"
              "    struct { int x : 4 = 1; } inner;\n"
              "#ifdef A\n"
              "    using enum E;\n"
              "#endif\n"
              "};\n")
assert Check.check(STORE, path, "C++17") == [
    "body[0].declarators[0].initializer.value.block.body[0]: CoReturnStatement needs C++20",
    "body[1].specifiers[0].members.members[0].specifiers[0].members.members[0].declarators[0]: MemberDeclarator with "
    "a bit-field's initializer needs C++20",
    "body[1].specifiers[0].members.members[1].directive.group.members[0]: UsingEnumDeclaration needs C++20"], (
    Check.check(STORE, path, "C++17"))
assert Check.check(STORE, path, "C++20") == [] and Check.validate(STORE, path) == []
# Validated as statements: a kind's rules, an `Error` and a term's problems within a block, a member list or a group
path = written(
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.lambdaexpression(lambda x: x.block(
        lambda k: k.compoundstatement(lambda k: k.body([lambda u: u.whilestatement(lambda s: s.condition(IS(a))),
                                                        lambda u: u.error(lambda e: e.text("x = = 1"))])))))),
    lambda u: u.directive(lambda s: s.directive(lambda d: d.ifdefdirective(lambda i: i.name(lambda n: n.spelling(
        "A")).group(lambda g: g.cpp_group(lambda g: g.body([lambda u: u.expressionstatement(lambda s: s.expression(
            BAD))])))))))
assert Check.validate(STORE, path) == [
    "body[0].expression.block.body[0]: a WhileStatement needs a body",
    "body[0].expression.block.body[1]: invalid syntax 'x = = 1'",
    "body[1].directive.group.body[0].expression: a memberexpression's operator must be '.' or '->', got '::'"]
# A term's dialect stops at its block (`Terms.boundaries`): what it holds is this module's to read
with STORE.transaction():
    unit = STORE.singleton(Model.WORKSPACE).files({"path": path})
    (_, statement), = Objects.properties(Objects.properties(unit)["body"].values[0]).items()
    expression = Objects.properties(statement)["expression"]
    assert Terms.kind_of(expression)[0] == "lambdaexpression" and CppExpressions.DIALECT.validate(expression) == []
print("ok")

## XCHK-10 · The ABI applied at the call: `validate` defers what depends on the target, `validate_for` checks it

As the user decided: "Of course C and C++ need a ABI, but if we can model programs in an abi-agnostic form and apply
the ABI at the last minute, that would be advantageous". A unit names no ABI, and a character literal holds its
escape's own number (`'\xc8'` is 200). `validate` defers whether it fits its type, `char` in C++ (where C's is `int`);
`validate_for` applies an ABI, C's `ABIS`, which serve C and C++ alike: x86-64's plain `char` is signed, which 200 does
not fit, and its `wchar_t` is `int`, which `L'\xffffffff'` does not fit; AArch64's are unsigned, which both fit. An ABI
not among `ABIS` is refused.

In [ ]:
from mbse.Programs import C

abi_store = Standards.prepare(Cpp.store())
for abi in Check.ABIS:
    Check.prepare_abi(abi_store, abi)
assert Check.ABIS == C.Check.ABIS and Check.prepare_abi is C.Check.prepare_abi  # C's, serving C++ alike
abi_store.transaction(Parse.parse("char c = '\\xc8';\nwchar_t w = L'\\xffffffff';\nchar d = '\\x7f';\n", "abi.cpp"))
assert Check.validate(abi_store, "abi.cpp") == []  # deferred: no ABI applied
assert Check.validate_for(abi_store, "abi.cpp", "Expressions.SystemV.AMD64") == [
    "body[0].declarators[0].initializer: integer literal 200 does not fit 'char'",
    "body[1].declarators[0].initializer: integer literal 4294967295 does not fit 'wchar_t'"]
assert Check.validate_for(abi_store, "abi.cpp", "Expressions.Arm.AAPCS64") == []
with abi_store.transaction():
    unit = abi_store.singleton(Model.WORKSPACE).files({"path": "abi.cpp"})
    assert Check.validate_file_for(abi_store, unit, "Expressions.SystemV.AMD64")[0].endswith("does not fit 'char'")
    with raises(LookupError, match="no ABI 'ILP32'"):
        Check.validate_file_for(abi_store, unit, "ILP32")
with raises(LookupError):
    Check.validate_for(abi_store, "missing.cpp", "Expressions.SystemV.AMD64")
print("ok")